<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata

from sqlalchemy import create_engine, text, inspect
engine = create_engine(userdata.get("NEON_DATABASE_URL"))

# Pulling data before the churn cut off point
query = """
CREATE OR REPLACE VIEW v_customer_metrics AS
SELECT "CustomerID",
    SUM("Revenue") AS "TotalRevenue",
    MIN("InvoiceDate") AS "FirstPurchase",
    MAX("InvoiceDate") AS "LastPurchase",
    MAX("InvoiceDate") - MIN("InvoiceDate") AS "Tenure",
    (SELECT MAX("InvoiceDate")
    FROM v_clean_sales_analytics) - MAX("InvoiceDate") AS "Recency",
    COUNT(DISTINCT "StockCode") AS "ProductDiversity",
    COUNT(DISTINCT "InvoiceDate") AS "Frequency",
    TO_CHAR(MIN("InvoiceDate"), 'YYYY-MM') AS "CohortMonth"
FROM v_clean_sales_analytics
WHERE "InvoiceDate" <= '2011-08-31'
GROUP BY "CustomerID"
"""

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

with engine.begin() as conn:
    conn.execute(text(query))
    print("Done! v_customer_metric view created/updated in postgres Database")

#Pulling and reviewing created view from database
query = """
SELECT *
FROM v_customer_metrics
LIMIT 5
"""

metrics_df = pd.read_sql(text(query), con = engine)
print('\nData Preview:')
metrics_df

Done! v_customer_metric view created/updated in postgres Database

Data Preview:


,CustomerID,TotalRevenue,FirstPurchase,LastPurchase,Tenure,Recency,ProductDiversity,Frequency,CohortMonth
0,12346.0,77183.60,2011-01-18,2011-01-18,0,325,1,1,2011-01
1,12347.0,2790.86,2010-12-07,2011-08-02,238,129,82,5,2010-12
2,12348.0,1487.24,2010-12-16,2011-04-05,110,248,22,3,2010-12
3,12350.0,334.40,2011-02-02,2011-02-02,0,310,17,1,2011-02
4,12352.0,1561.81,2011-02-16,2011-03-22,34,262,26,4,2011-02


In [21]:
engine = create_engine(userdata.get("NEON_DATABASE_URL"))

query = """
CREATE OR REPLACE VIEW v_customer_rfm AS
WITH metrics AS
(SELECT "CustomerID",
    (SELECT MAX("InvoiceDate")
    FROM v_clean_sales_analytics) - MAX("InvoiceDate") AS "Recency",
    COUNT(DISTINCT "InvoiceDate") AS "Frequency",
    SUM("Revenue") AS "Monetary"
FROM v_clean_sales_analytics
GROUP BY "CustomerID"),

rfm_scores AS
(SELECT "CustomerID",
    NTILE(5) OVER(ORDER BY "Recency" DESC) AS "R",
    NTILE(5) OVER(ORDER BY "Frequency" ASC) AS "F",
    NTILE(5) OVER(ORDER BY "Monetary" ASC) AS "M"
FROM metrics)

SELECT *,
    "R" + "F" + "M" AS "RfmScore"
FROM rfm_scores
"""

with engine.begin() as conn:
    conn.execute(text(query))
    print("Done! Successfully created/updated v_customer_metrics view in Postgres database")

#Preview of created view
rfm_query = """
SELECT *
FROM v_customer_rfm;
"""
rfm_df = pd.read_sql(text(rfm_query), con = engine)
print('\nData Preview:')
rfm_df.head()

Done! Successfully created/updated v_customer_metrics view in Postgres database

Data Preview:


,CustomerID,R,F,M,RfmScore
0,17908.0,1,1,1,3
1,15350.0,1,2,1,4
2,16274.0,1,2,2,5
3,12791.0,1,1,1,3
4,14142.0,1,2,2,5
